In [45]:
# setup imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from plotly.io import show

PROJECT_ROOT = Path().resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import warnings
import optuna
import optunahub
import seaborn as sns
from dask import delayed
from dask.distributed import Client, progress
from ewatercycle.observation.grdc import get_grdc_data

from src.forcing import generate_lumped_ERA5_forcing, load_lumped_forcing_data
from src.models import make_objective_safe,wrap_objective_safe_unscaled
from src.paths import *
from src.constants import PARAMETER_NAMES, HBV_PARAM_BOUNDS

warnings.filterwarnings("ignore", category=UserWarning)

# Keep imports available for later cells without linter "unused import" warnings.
_ = (plt, np, pd, sns, delayed, Client, progress)

In [46]:
shape_name = "Chatly_GRDC"
start_date = "1955-01-02T00:00Z"
end_date = "1959-12-31T00:00Z"

#generate_lumped_ERA5_forcing(shape_name, start=start_date, end=end_date)
ERA5_forcing_loaded = load_lumped_forcing_data("Chatly_GRDC", "ERA5", "1955-1959")

grdc_station = get_grdc_data(
    2817100, "1955-01-02T00:00Z", "1959-12-31T00:00Z", data_home=GRDC / "Daily"
)

q_obs = grdc_station["streamflow"]

In [47]:
#objective_fn = make_objective_safe(ERA5_forcing_loaded, q_obs.values, shape_name)
objective_fn = wrap_objective_safe_unscaled(ERA5_forcing_loaded, q_obs.values, shape_name)

In [48]:
n_trials = 220
n_generations = 10

n_jobs = 1
optuna_seed = 42
study_name = "optuna_test_16april_TPE_v2"
study_name_fixed = "optuna_test_16april_TPE"
show_progress_bar = True

In [49]:

#storage_name = f"sqlite:///{study_name}.db"
storage_name = f"sqlite:///{study_name_fixed}.db"


#sampler = optuna.samplers.GPSampler(seed=optuna_seed)
sampler = optuna.samplers.TPESampler(multivariate=True, seed=optuna_seed)
# module = optunahub.load_module(
#      package="samplers/nelder_mead",
# )

# # study.optimize can be used with an Optuna-style objective function.
# sampler = module.NelderMeadSampler(seed=123)

# sampler = optuna.samplers.GPSampler(seed=optuna_seed)

# module = optunahub.load_module("samplers/cma_es_refinement")
# sampler = module.CmaEsRefinementSampler(seed=42)




# sampler=optunahub.load_module(
#         "samplers/auto_sampler"
#     ).AutoSampler(seed=optuna_seed)


# sampler = optunahub.load_module(package="samplers/pso").PSOSampler(
#     n_particles=int(n_trials / n_generations),
#     inertia=0.5,
#     cognitive=1.5,
#     social=1.5,
# )


/tmp/ipykernel_2296690/486417462.py:6: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.


In [50]:

local_history = {}
if study_name is None:
    study_name = f"optuna_seed_{optuna_seed}"


study = optuna.create_study(direction="minimize", sampler=sampler, study_name=study_name_fixed,storage=storage_name, load_if_exists=True)

def objective_wrapped(trial):
    theta_phys = np.array([
        trial.suggest_float(name, float(pmin), float(pmax))
        for name, pmin, pmax in zip(
            PARAMETER_NAMES, HBV_PARAM_BOUNDS["min"], HBV_PARAM_BOUNDS["max"]
        )
    ])
    return float(objective_fn(theta_phys))

study.optimize(
    objective_wrapped,
    n_trials=n_trials,
    n_jobs=1,
    show_progress_bar=show_progress_bar,
   )

[I 2026-04-16 17:00:01,929] Using an existing study with name 'optuna_test_16april_TPE' instead of creating a new one.


  0%|          | 0/220 [00:00<?, ?it/s]

[I 2026-04-16 17:00:09,321] Trial 200 finished with value: 0.40376189612378655 and parameters: {'Imax': 10.069486506525315, 'Ce': 0.9858551408192854, 'Sumax': 752.9871293588279, 'Beta': 1.169713403204896, 'Pmax': 0.11333089189864658, 'Tlag': 12.94529523323101, 'Kf': 0.015415203937011828, 'Ks': 0.003939527820354856, 'FM': 0.059381076540495153}. Best is trial 113 with value: 0.213213051831801.
[I 2026-04-16 17:00:16,484] Trial 201 finished with value: 0.4429368469377148 and parameters: {'Imax': 5.4739658072154, 'Ce': 0.7064800602093458, 'Sumax': 545.3400273481863, 'Beta': 1.8330210241323264, 'Pmax': 0.11979649953758151, 'Tlag': 10.691732085827233, 'Kf': 0.01361985452773386, 'Ks': 0.0023355350774566445, 'FM': 0.3008231583035762}. Best is trial 113 with value: 0.213213051831801.
[I 2026-04-16 17:00:23,960] Trial 202 finished with value: 0.5611260496762797 and parameters: {'Imax': 6.95886674669026, 'Ce': 0.9717815466173039, 'Sumax': 757.4499926769073, 'Beta': 2.3700463083762817, 'Pmax': 0.1

In [51]:
fig = optuna.visualization.plot_optimization_history(study)
show(fig)

In [52]:

fig2 = optuna.visualization.plot_param_importances(study)
show(fig2)

In [53]:
fig3 = optuna.visualization.plot_contour(study, params=["Ce", "Beta"])
show(fig3)

In [54]:
fig4 = optuna.visualization.plot_slice(study) #, params=["Ce", "Beta", "FM", "Pmax", "Sumax","Tlag"])
show(fig4)

In [55]:
print(study)

In [56]:
study.best_params

{'Imax': 14.475301510733365,
 'Ce': 0.9630412176801041,
 'Sumax': 512.5278771355537,
 'Beta': 1.7331614494100278,
 'Pmax': 0.23839835054346947,
 'Tlag': 7.915325000147259,
 'Kf': 0.01640487236680185,
 'Ks': 0.009777106273188018,
 'FM': 0.12197640546773336}

In [57]:
df_test = study.trials_dataframe(attrs=("number", "value", "params", "state"))

In [58]:
df_test.to_csv(f"{study.study_name}_trials.csv", index=False)

In [59]:
fig = optuna.visualization.plot_rank(study,["Beta", "Sumax"])
show(fig)

In [60]:
fig = optuna.visualization.plot_parallel_coordinate(study) #, params=["Beta", "Sumax", "Ce", "FM"])
show(fig)